In [2]:
# ==========================================
# STEP 1: IMPORT LIBRARIES AND CONNECT DRIVE
# ==========================================

# Import libraries required for data processing
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set the project directory
project_path = '/content/drive/MyDrive/Wafer Classification'

# Specify the original dataset path
dataset_path = os.path.join(project_path, 'LSWMD.pkl')

# Load the original wafer dataset
df = pd.read_pickle(dataset_path)

# Display dataset information
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

# Check GPU availability
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
Dataset loaded successfully!
Dataset shape: (811457, 6)
Device: cpu


In [3]:
# ==========================================
# STEP 2: LOAD EXISTING DATASET SPLITS
# ==========================================

# Define the paths of the saved split files
splits_path = os.path.join(project_path, 'splits')

train_df = pd.read_csv(os.path.join(splits_path, 'train.csv'))
val_df = pd.read_csv(os.path.join(splits_path, 'validation.csv'))
test_df = pd.read_csv(os.path.join(splits_path, 'test.csv'))

# Display the number of samples
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

# Display the first five training records
print("\nTraining split preview:")
print(train_df.head())

# Check whether all three splits were loaded
print("\nAll splits loaded successfully!")

Training samples: 120979
Validation samples: 26116
Testing samples: 25855

Training split preview:
   row_index lotName  label_id label
0         50    lot3         8  none
1         51    lot3         8  none
2         52    lot3         8  none
3         53    lot3         8  none
4         54    lot3         8  none

All splits loaded successfully!


In [4]:
# ==========================================
# STEP 3: VERIFY DATASET INDICES
# ==========================================

# Get the original row index of the first training sample
idx = int(train_df.iloc[0]['row_index'])

# Compare the lot name in the split file
print("Split CSV lot name:", train_df.iloc[0]['lotName'])

# Check the original DataFrame using label-based indexing
print("Using df.loc:", df.loc[idx, 'lotName'])

# Check the original DataFrame using positional indexing
print("Using df.iloc:", df.iloc[idx]['lotName'])

# Display the saved label
print("Saved class label:", train_df.iloc[0]['label'])

# Display the saved class ID
print("Saved class ID:", train_df.iloc[0]['label_id'])

Split CSV lot name: lot3
Using df.loc: lot3
Using df.iloc: lot3
Saved class label: none
Saved class ID: 8


In [6]:
# ==========================================
# STEP 1: DEFINE LABEL EXTRACTION FUNCTION
# ==========================================

# Convert the failureType array into a readable class label

import numpy as np

def extract_label(x):

    arr = np.asarray(x).flatten()

    if arr.size == 0:
        return "Unknown"

    return str(arr[0])


print("Label extraction function defined successfully!")

Label extraction function defined successfully!


In [7]:
# ==========================================
# STEP 4: VERIFY ALL DATASET SPLIT RECORDS
# ==========================================

# Check whether every saved row index matches
# the original DataFrame's lot name and label

def verify_split(split_df, split_name):

    correct = 0
    errors = 0

    for _, row in split_df.iterrows():

        # Get the original row position
        idx = int(row['row_index'])

        # Retrieve the original wafer record
        original_row = df.iloc[idx]

        # Extract the original class label
        original_label = extract_label(original_row['failureType'])

        # Compare lot name and class label
        if (original_row['lotName'] == row['lotName']
                and original_label == row['label']):
            correct += 1
        else:
            errors += 1

    print(f"\n{split_name} split:")
    print("Correct records:", correct)
    print("Incorrect records:", errors)

    return errors == 0


# Verify all three splits
train_valid = verify_split(train_df, "Training")
val_valid = verify_split(val_df, "Validation")
test_valid = verify_split(test_df, "Testing")

print("\nAll splits aligned:",
      train_valid and val_valid and test_valid)


Training split:
Correct records: 120979
Incorrect records: 0

Validation split:
Correct records: 26116
Incorrect records: 0

Testing split:
Correct records: 25855
Incorrect records: 0

All splits aligned: True


In [8]:
# =====================================================
# STEP 1: PREPROCESS WAFER MAPS FOR RESNET-18
# =====================================================

# Import required libraries
import numpy as np
from PIL import Image

# Function to resize each wafer map to 64 x 64
# while preserving its original aspect ratio.

def preprocess_wafer(wafer, target_size=64):

    height, width = wafer.shape

    # Calculate scaling factor
    scale = min(target_size / height, target_size / width)

    new_height = max(1, round(height * scale))
    new_width = max(1, round(width * scale))

    # Resize using nearest-neighbour interpolation
    image = Image.fromarray(wafer.astype(np.uint8))

    resized = image.resize(
        (new_width, new_height),
        Image.Resampling.NEAREST
    )

    resized = np.array(resized)

    # Create a blank 64 x 64 canvas
    canvas = np.zeros(
        (target_size, target_size),
        dtype=np.uint8
    )

    # Calculate centre position
    top = (target_size - new_height) // 2
    left = (target_size - new_width) // 2

    # Place resized wafer map in the centre
    canvas[
        top:top + new_height,
        left:left + new_width
    ] = resized

    return canvas


print("Wafer preprocessing function created successfully!")

Wafer preprocessing function created successfully!


In [9]:
# =====================================================
# STEP 2: PROCESS ALL THREE DATASET SPLITS
# =====================================================

# Extract and preprocess wafer maps using original row indices

def preprocess_split(split_df, split_name):

    images = []

    indices = split_df["row_index"].values

    for i, idx in enumerate(indices):

        # Retrieve original wafer map
        wafer = df.loc[idx, "waferMap"]

        # Preprocess wafer map
        processed = preprocess_wafer(wafer)

        images.append(processed)

        if (i + 1) % 10000 == 0:
            print(f"{split_name}: {i + 1}/{len(indices)}")

    return np.stack(images)


# Process each split separately
X_train = preprocess_split(train_df, "Training")
X_val = preprocess_split(val_df, "Validation")
X_test = preprocess_split(test_df, "Testing")


# Display final shapes
print("\nPreprocessing completed!")

print("Training images:", X_train.shape)
print("Validation images:", X_val.shape)
print("Testing images:", X_test.shape)

print("Data type:", X_train.dtype)

Training: 10000/120979
Training: 20000/120979
Training: 30000/120979
Training: 40000/120979
Training: 50000/120979
Training: 60000/120979
Training: 70000/120979
Training: 80000/120979
Training: 90000/120979
Training: 100000/120979
Training: 110000/120979
Training: 120000/120979
Validation: 10000/26116
Validation: 20000/26116
Testing: 10000/25855
Testing: 20000/25855

Preprocessing completed!
Training images: (120979, 64, 64)
Validation images: (26116, 64, 64)
Testing images: (25855, 64, 64)
Data type: uint8


In [10]:
# =====================================================
# STEP 3: PREPARE NUMERICAL LABELS
# =====================================================

# Extract class labels from the saved CSV splits
# Each label_id represents one of the nine defect classes.

y_train = train_df["label_id"].values.astype(np.int64)
y_val = val_df["label_id"].values.astype(np.int64)
y_test = test_df["label_id"].values.astype(np.int64)

# Define the class names in the correct numerical order
class_names = [
    "Center",
    "Donut",
    "Edge-Loc",
    "Edge-Ring",
    "Loc",
    "Near-full",
    "Random",
    "Scratch",
    "none"
]

# Display label shapes
print("Training labels:", y_train.shape)
print("Validation labels:", y_val.shape)
print("Testing labels:", y_test.shape)

# Verify the number of unique classes
print("\nUnique training labels:", np.unique(y_train))
print("Number of classes:", len(class_names))

# Verify that image and label counts match
print("\nTraining images and labels match:",
      len(X_train) == len(y_train))

print("Validation images and labels match:",
      len(X_val) == len(y_val))

print("Testing images and labels match:",
      len(X_test) == len(y_test))

Training labels: (120979,)
Validation labels: (26116,)
Testing labels: (25855,)

Unique training labels: [0 1 2 3 4 5 6 7 8]
Number of classes: 9

Training images and labels match: True
Validation images and labels match: True
Testing images and labels match: True


In [11]:
# =====================================================
# STEP 4: CALCULATE CLASS WEIGHTS
# =====================================================

# Import the function used to calculate balanced class weights
from sklearn.utils.class_weight import compute_class_weight

# Calculate weights using ONLY the training labels
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

# Display the calculated weights
print("ResNet-18 Class Weights:\n")

for name, weight in zip(class_names, class_weights):
    print(f"{name}: {weight:.4f}")

ResNet-18 Class Weights:

Center: 4.4688
Donut: 36.7271
Edge-Loc: 3.7031
Edge-Ring: 1.9564
Loc: 5.5753
Near-full: 125.6272
Random: 21.8927
Scratch: 15.4330
none: 0.1304


In [12]:
# =====================================================
# STEP 5: CREATE PYTORCH DATASET CLASS
# =====================================================

# Import PyTorch libraries
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# Define a custom Dataset class for wafer maps
class WaferDataset(Dataset):

    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    # Return the total number of samples
    def __len__(self):
        return len(self.images)

    # Retrieve one image and its corresponding label
    def __getitem__(self, index):

        # Convert wafer map into a PyTorch tensor
        image = torch.tensor(
            self.images[index],
            dtype=torch.long
        )

        # Convert categorical values into 3 channels
        image = F.one_hot(
            image,
            num_classes=3
        )

        # Change dimensions from (64, 64, 3) to (3, 64, 64)
        image = image.permute(2, 0, 1).float()

        # Retrieve corresponding class label
        label = torch.tensor(
            self.labels[index],
            dtype=torch.long
        )

        return image, label


print("WaferDataset class created successfully!")

WaferDataset class created successfully!


In [13]:
# =====================================================
# STEP 6: CREATE TRAINING, VALIDATION AND TEST DATASETS
# =====================================================

# Create separate Dataset objects for all three splits
train_dataset = WaferDataset(X_train, y_train)
val_dataset = WaferDataset(X_val, y_val)
test_dataset = WaferDataset(X_test, y_test)

# Display the number of samples
print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Testing dataset:", len(test_dataset))

Training dataset: 120979
Validation dataset: 26116
Testing dataset: 25855


In [14]:
# =====================================================
# STEP 7: CREATE PYTORCH DATALOADERS
# =====================================================

# Set batch size
batch_size = 128

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Display the number of batches
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 946
Validation batches: 205
Testing batches: 202


In [15]:
# =====================================================
# STEP 8: VERIFY ONE TRAINING BATCH
# =====================================================

# Retrieve one batch from the training DataLoader
images, labels = next(iter(train_loader))

# Display image and label dimensions
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

# Display data types
print("Image data type:", images.dtype)
print("Label data type:", labels.dtype)

# Check unique pixel values and labels
print("Unique pixel values:", torch.unique(images))
print("Unique class labels:", torch.unique(labels))

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Image batch shape: torch.Size([128, 3, 64, 64])
Label batch shape: torch.Size([128])
Image data type: torch.float32
Label data type: torch.int64
Unique pixel values: tensor([0., 1.])
Unique class labels: tensor([0, 2, 3, 4, 7, 8])


In [17]:
# =====================================================
# STEP 9: CHECK GPU AVAILABILITY
# =====================================================

# Check whether CUDA GPU is available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Selected device:", device)

# Display GPU name if available
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.14.1+cpu
GPU available: False
Selected device: cpu
